# 03 — Inventory Optimization

In [1]:
%cd /content
!test -d demand-forecasting-inventory-optimization || git clone https://github.com/hemanthkarthikd/demand-forecasting-inventory-optimization.git
%cd /content/demand-forecasting-inventory-optimization/notebooks

/content
Cloning into 'demand-forecasting-inventory-optimization'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 29 (delta 6), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 52.49 KiB | 440.00 KiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/demand-forecasting-inventory-optimization/notebooks


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import norm

demand = pd.read_csv("../data/raw/weekly_demand.csv", parse_dates=["week_start"])
sku = pd.read_csv("../data/raw/sku_master.csv")
profile = pd.read_csv("../data/processed/sku_demand_profile.csv")

In [3]:
stats = demand.groupby("sku_id")["demand_units"].agg(
    avg_weekly_demand="mean", demand_std="std"
).reset_index()

inv = sku.merge(stats,on="sku_id").merge(
    profile[["sku_id","abc_class","xyz_class"]], on="sku_id"
)

inv["annual_demand"] = inv["avg_weekly_demand"]*52
inv["annual_holding_cost_per_unit"] = inv["unit_cost"]*inv["annual_holding_rate"]
inv["eoq_units"] = np.sqrt(
    (2*inv["annual_demand"]*inv["ordering_cost"]) /
    inv["annual_holding_cost_per_unit"]
)
inv["z_score"] = inv["target_service_level"].apply(norm.ppf)
inv["safety_stock_units"] = inv["z_score"]*inv["demand_std"]*np.sqrt(inv["lead_time_weeks"])
inv["reorder_point_units"] = inv["avg_weekly_demand"]*inv["lead_time_weeks"] + inv["safety_stock_units"]
inv["annual_ordering_cost"] = (inv["annual_demand"]/inv["eoq_units"])*inv["ordering_cost"]
inv["annual_cycle_stock_holding_cost"] = (inv["eoq_units"]/2)*inv["annual_holding_cost_per_unit"]
inv["annual_safety_stock_holding_cost"] = inv["safety_stock_units"]*inv["annual_holding_cost_per_unit"]
inv["estimated_annual_inventory_cost"] = inv[
    ["annual_ordering_cost","annual_cycle_stock_holding_cost","annual_safety_stock_holding_cost"]
].sum(axis=1)

for c in ["eoq_units","safety_stock_units","reorder_point_units"]:
    inv[c] = inv[c].round(0).astype(int)

inv.to_csv("../data/processed/inventory_policy.csv", index=False)
display(inv[[
    "sku_id","sku_name","abc_class","xyz_class","lead_time_weeks",
    "target_service_level","eoq_units","safety_stock_units",
    "reorder_point_units","estimated_annual_inventory_cost"
]])

,sku_id,sku_name,abc_class,xyz_class,lead_time_weeks,target_service_level,eoq_units,safety_stock_units,reorder_point_units,estimated_annual_inventory_cost
0,SKU-001,Electronics Item 1,C,X,5,0.95,443,77,563,3984.060646
1,SKU-002,Electronics Item 2,A,X,3,0.97,456,262,1133,10279.666142
2,SKU-003,Electronics Item 3,A,X,5,0.95,295,237,1827,19673.153768
3,SKU-004,Electronics Item 4,A,X,4,0.95,216,148,1123,17428.552591
4,SKU-005,Electronics Item 5,A,X,3,0.95,408,140,737,7100.148574
5,SKU-006,Electronics Item 6,A,Z,3,0.90,181,339,882,26855.751139
6,SKU-007,Fasteners Item 1,B,X,5,0.95,398,124,1134,7049.605581
7,SKU-008,Fasteners Item 2,C,X,6,0.95,751,187,1544,3047.969861
8,SKU-009,Fasteners Item 3,A,X,5,0.95,360,184,1377,7585.199646
9,SKU-010,Fasteners Item 4,A,X,6,0.95,210,170,1436,14870.283271
